# COMMONWEAVE Zero — Validación · Performance · Veredicto de eficiencia

Notebook autónomo para:

1. **Validación** — equivalencia semántica del adaptador generado vs referencia Python  
2. **Medición de performance** — latencia (mediana / p99) en cuatro implementaciones  
3. **Veredicto de eficiencia** — esfuerzo humano vs rendimiento vs corrección  

**Requisitos**

```bash
pip install pyyaml cryptography
# gcc + make para compilar libmatmul.so
```

Coloca este notebook dentro de `upstream/commonweave_zero/` (o ajusta `ROOT` en la siguiente celda).

In [ ]:
from __future__ import annotations

import json
import statistics
import subprocess
import sys
import time
from pathlib import Path

# --- localizar raíz del prototipo ---
CANDIDATES = [
    Path(".").resolve(),
    Path("..").resolve(),
    Path("../upstream/commonweave_zero").resolve(),
    Path("upstream/commonweave_zero").resolve(),
]
ROOT = None
for c in CANDIDATES:
    if (c / "run_zero.py").exists() or (c / "contracts" / "matmul.yaml").exists():
        ROOT = c
        break
if ROOT is None:
    raise FileNotFoundError(
        "No se encuentra commonweave_zero. Coloca el notebook dentro del directorio del prototipo."
    )

sys.path.insert(0, str(ROOT))
print(f"ROOT = {ROOT}")
print(f"Python {sys.version.split()[0]}")

## 0. Compilar nativo y regenerar adaptador

In [ ]:
def sh(cmd, cwd=ROOT):
    print("+", " ".join(str(x) for x in cmd))
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout)
        print(r.stderr)
        raise RuntimeError(f"comando falló: {cmd}")
    if r.stdout.strip():
        print(r.stdout.strip()[:500])
    return r

sh(["make"], cwd=ROOT / "c_src")
sh([
    sys.executable, str(ROOT / "generator" / "generate_adapter.py"),
    "--contract", str(ROOT / "contracts" / "matmul.yaml"),
    "--lib", str(ROOT / "c_src" / "libmatmul.so"),
    "--out", str(ROOT / "generated" / "matmul_adapter.py"),
])
print("✓ nativo + adaptador listos")

## 1. Validación diferencial

Compara el adaptador generado contra el modelo de referencia en Python puro  
(entradas aleatorias + casos límite de dimensión inválida).

In [ ]:
diff_out = ROOT / "report" / "differential_nb.json"
diff_out.parent.mkdir(parents=True, exist_ok=True)

sh([
    sys.executable, str(ROOT / "verifier" / "differential.py"),
    "--trials", "80",
    "--seed", "20261019",
    "--out", str(diff_out),
])

diff = json.loads(diff_out.read_text())
print("\n=== RESULTADO VALIDACIÓN ===")
print(f"  veredicto   : {diff['verdict']}")
print(f"  pass_rate   : {diff['pass_rate']:.1%}  ({diff['passed']}/{diff['trials']})")
print(f"  edge cases  : {diff['edge_passed']}/{diff['edge_total']}")
assert diff["verdict"] == "PASS", "Validación diferencial FALLÓ"
print("✓ Validación PASS")

## 2. Medición de performance (cuatro vías)

| Implementación | Qué mide |
|----------------|----------|
| pure_python | reescritura completa (baseline de coste de no integrar) |
| manual_ctypes | binding convencional escrito a mano |
| generated | adaptador emitido por COMMONWEAVE desde el contrato |
| ipc | proceso separado + Pipe (aislamiento) |

In [ ]:
bench_out = ROOT / "report" / "benchmark_nb.json"

sh([
    sys.executable, str(ROOT / "benchmarks" / "bench_four_way.py"),
    "--sizes", "4,8,16,32,48",
    "--repeats", "40",
    "--out", str(bench_out),
])

bench = json.loads(bench_out.read_text())

print("\n=== LATENCIA MEDIANA (µs) ===")
header = f"{'Size':>8}  {'pure_py':>10}  {'manual':>10}  {'generated':>10}  {'ipc':>10}  {'gen/manual':>10}"
print(header)
print("-" * len(header))

rows = []
for size, block in bench["implementations"].items():
    pure = block["pure_python"]["median"] * 1e6
    man = block["manual_ctypes"]["median"] * 1e6
    gen = block["generated"]["median"] * 1e6
    ipc = block["ipc"]["median"] * 1e6
    ratio = gen / man if man > 0 else float("nan")
    rows.append((int(size), pure, man, gen, ipc, ratio))
    print(f"{size + 'x' + size:>8}  {pure:10.1f}  {man:10.1f}  {gen:10.1f}  {ipc:10.1f}  {ratio:10.2f}x")

print("\n=== p99 (µs) — cola de latencia ===")
print(f"{'Size':>8}  {'pure_py':>10}  {'manual':>10}  {'generated':>10}  {'ipc':>10}")
for size, block in bench["implementations"].items():
    print(
        f"{size + 'x' + size:>8}  "
        f"{block['pure_python']['p99']*1e6:10.1f}  "
        f"{block['manual_ctypes']['p99']*1e6:10.1f}  "
        f"{block['generated']['p99']*1e6:10.1f}  "
        f"{block['ipc']['p99']*1e6:10.1f}"
    )

## 3. Veredicto de eficiencia

Criterios (alineados con el documento COMMONWEAVE):

- **Corrección**: validación diferencial PASS  
- **Esfuerzo**: LOC de pegamento (generated = 0 vs manual ≈ 25)  
- **Rendimiento**: latencia generada ≤ 1.25× manual en tamaños ≥ 8  
- **Aislamiento**: IPC es referencia de coste de proceso separado, no competidor directo

In [ ]:
effort = bench.get("effort_proxy", {})
loc_gen = effort.get("generated", {}).get("loc_glue", 0)
loc_man = effort.get("manual_ctypes", {}).get("loc_glue", 25)

# ratios gen/manual en tamaños >= 8
ratios_ge8 = [r for r in rows if r[0] >= 8]
ratios = [r[5] for r in rows_ge8]
max_ratio = max(ratios) if ratios else float("inf")
median_ratio = statistics.median(ratios) if ratios else float("inf")

# speedup vs pure python (tamaño 32 si existe, si no el mayor)
row32 = next((r for r in rows if r[0] == 32), rows[-1] if rows else None)
speedup_vs_pure = (row32[1] / row32[3]) if row32 and row32[3] > 0 else float("nan")

correct = diff["verdict"] == "PASS"
effort_ok = loc_gen < loc_man
perf_ok = max_ratio <= 1.25  # generado no más de 25% más lento que manual

print("=== CRITERIOS ===")
print(f"  Corrección (diferencial PASS)     : {'SÍ' if correct else 'NO'}  [{diff['verdict']}]")
print(f"  Esfuerzo (gen LOC < manual LOC)   : {'SÍ' if effort_ok else 'NO'}  [gen={loc_gen}, manual={loc_man}]")
print(f"  Performance (gen ≤ 1.25× manual)  : {'SÍ' if perf_ok else 'NO'}  [max ratio={max_ratio:.2f}×, mediana={median_ratio:.2f}×]")
print(f"  Speedup vs pure Python (ref size) : {speedup_vs_pure:.1f}×")

if correct and effort_ok and perf_ok:
    verdict = "EFICIENTE"
    detail = (
        "El adaptador generado es correcto, exige menos esfuerzo humano que un binding "
        "convencional y mantiene rendimiento competitivo (≤ 1.25× el manual en el camino caliente)."
    )
elif correct and effort_ok and not perf_ok:
    verdict = "CORRECTO PERO COSTOSO"
    detail = (
        "Corrección y menor esfuerzo humano, pero la latencia del generado supera el umbral "
        "de 1.25× respecto al binding manual. Revisar overhead de conversión ctypes."
    )
elif correct and not effort_ok:
    verdict = "CORRECTO SIN VENTAJA DE ESFUERZO"
    detail = "La integración es correcta pero no reduce el esfuerzo humano frente al baseline."
else:
    verdict = "NO APTO"
    detail = "Falló la validación semántica o no cumple los umbrales mínimos."

print("\n" + "=" * 60)
print(f"VEREDICTO DE EFICIENCIA: {verdict}")
print("=" * 60)
print(detail)
print()
print("Decisión COMMONWEAVE:", "PROCEED" if verdict == "EFICIENTE" else "REVISAR / NO PROCEED")

## 4. (Opcional) Verificar sello de integridad CSG

Si ya corriste `run_zero.py` y existe `report/sello.json`, valida firma + digests.

In [ ]:
sello_path = ROOT / "report" / "sello.json"
if sello_path.is_file():
    from seal.sello import SelloIntegridad
    data = json.loads(sello_path.read_text())
    vr = SelloIntegridad.verificar(data, base_dir=ROOT)
    print(json.dumps(vr, indent=2))
    if vr["signature_valid"] and vr["artifacts_intact"]:
        print("✓ Sello CSG válido")
    else:
        print("✗ Sello inválido o artefactos alterados")
else:
    print("No hay report/sello.json — ejecuta antes: python run_zero.py")

## 5. Resumen exportable

In [ ]:
summary = {
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "validation": {
        "verdict": diff["verdict"],
        "pass_rate": diff["pass_rate"],
        "edge": f"{diff['edge_passed']}/{diff['edge_total']}",
    },
    "performance": {
        "sizes": [r[0] for r in rows],
        "median_us": {
            "pure_python": [round(r[1], 1) for r in rows],
            "manual_ctypes": [round(r[2], 1) for r in rows],
            "generated": [round(r[3], 1) for r in rows],
            "ipc": [round(r[4], 1) for r in rows],
        },
        "gen_over_manual_max": round(max_ratio, 3),
        "gen_over_manual_median": round(median_ratio, 3),
        "speedup_vs_pure_ref": round(speedup_vs_pure, 2),
    },
    "effort_loc_glue": {"generated": loc_gen, "manual_ctypes": loc_man},
    "efficiency_verdict": verdict,
    "decision": "PROCEED" if verdict == "EFICIENTE" else "REVISAR",
}

out_summary = ROOT / "report" / "veredicto_eficiencia.json"
out_summary.write_text(json.dumps(summary, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(summary, indent=2, ensure_ascii=False))
print(f"\nGuardado en {out_summary}")